In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

## Settings

In [ ]:
from src.communication import LunanetSatAntenna
import numpy as np
import matplotlib.pyplot as plt

a = 9750e3
e = 0.6383
i = np.deg2rad(40)
RAAN = 0
w = 90
M = 0
coe = np.array([a, e, i, RAAN, w, M])

ant = LunanetSatAntenna(P_tx=15.0, coe=coe)

ant.plot_pattern()
plt.savefig("figs/antenna_pattern_ref.pdf", dpi=300)

In [ ]:
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

# constants
n_sma = 13
n_incs = 13
n_orbit = 4  # number of orbit to simulate

# results
results = np.ones((n_sma, n_incs, 3)) * np.nan  # RMS, 95%, 99.7% (m)

stats_ratio = (n_orbit - 1) / n_orbit  # ratio of the stats period

# initial conditions
smas = np.linspace(4000, 16000, n_sma) * 1e3
incs = np.deg2rad(np.linspace(40, 70, n_incs))

In [ ]:
from src.gridsearch import gridsearch_receiver_noise
import os

recalc = True  # if True, recalculate the gridsearch, otherwise load existing results
P_tx_dBw = 8.0  # Transmit power in dBW
CN0_thresh = 30.0  # Minimum CN0 threshold in dB-Hz

filename = "data/gridsearch_noise/gridsearch_noise_results_Ptx_{}_CNT_{}.npy".format(
    int(P_tx_dBw), int(CN0_thresh)
)

os.makedirs("data/gridsearch_noise", exist_ok=True)

if os.path.exists(filename) and not recalc:
    results = np.load(filename)
    print("Loaded existing results from", filename)
else:
    results = gridsearch_receiver_noise(
        et0, smas, incs, P_tx_dBw=P_tx_dBw, min_elev_deg=5.0, CN0_thresh=CN0_thresh
    )

    # save results
    np.save(filename, results)

In [ ]:
np.save(filename, results)

In [ ]:
# 2d contour plot (sma vs inc)
validx = (
    3  # 0: RMS, 1: 95%, 2: 99.7%  3: CN0　(RMS, over 20, over 30 in dB-Hz CN0 in dB-Hz)
)

for validx in [1, 5]:
    labels = ["RMS", "95p", "99.7p", "CN0", "over 20 dB-Hz", "over 30 dB-Hz"]
    metrics = "URE" if validx <= 2 else "CN0"
    smas_km = smas / 1e3  # convert to km

    fig = plt.figure(figsize=(7, 5))
    incd = np.rad2deg(incs)
    igrid, agrid = np.meshgrid(incd, smas_km)
    plt.contourf(
        incd, smas_km, results[:, :, validx], cmap="viridis", levels=50, alpha=0.3
    )
    plt.scatter(igrid, agrid, c=results[:, :, validx], cmap="viridis")
    # print the text of coverage
    for i in range(len(smas)):
        for j in range(len(incd)):
            eps_h = 200
            eps_i = 0.5
            if results[i, j, validx] > 0:
                plt.text(
                    incd[j] + eps_i,
                    smas_km[i] + eps_h,
                    f"{results[i, j, validx]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=9,
                )
    # # plot the minimum altitude point
    # plt.scatter(max_cov_incs, [min_alt + pnt.R_MOON] * len(max_cov_incs), c="red", marker="x", label="Minimum altitude")
    # plt.colorbar()
    plt.xlabel("Inclination [deg]", fontsize=14, fontweight="bold")
    plt.ylabel("Semi-major axis [km]", fontsize=14, fontweight="bold")
    # set minor grid to hlist and ilist
    plt.grid()
    plt.xticks(incd)
    plt.yticks(smas_km)
    plt.ylim(pnt.R_MOON * 1e-3, smas_km[-1] + 500)
    plt.xlim(incd[0] - 0.5, incd[-1] + 0.5)
    plt.savefig(
        "figs/noise/{}_{}_gridsearch_sma_inc_Ptx_{}_CN0_{}.pdf".format(
            metrics, labels[validx], int(P_tx_dBw), int(CN0_thresh)
        ),
        dpi=300,
    )
    plt.show()